# 📊 Evaluación Comparativa de Modelos de Embeddings para RAG

## 🎯 Objetivo del Análisis

El presente cuaderno evalúa exhaustivamente **5 modelos de embeddings multilingües** sobre un corpus de documentación bancaria, con el objetivo de seleccionar el modelo óptimo para un sistema **RAG (Retrieval-Augmented Generation)** en producción.

Se analizan tres dimensiones complementarias:

| Dimensión | Métricas |
|-----------|----------|
| **Precisión semántica** | MRR, Top-1 Accuracy, Top-3 Accuracy |
| **Eficiencia computacional** | Tiempo de embedding, Consumo de RAM |
| **Estrategia de chunking** | Chunk Size (500 / 1000 / 1500), Overlap (50 / 150 / 300) |

> **45 combinaciones** evaluadas: 5 modelos × 3 chunk sizes × 3 overlaps = 45 configuraciones completas.

---
## ⚙️ 0. Configuración del Entorno

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.templates.default = 'plotly_white'

# ── Paleta de colores corporativa ──────────────────────────────────────────
PALETTE = [
    '#1F3864',  # azul oscuro
    '#2E75B6',  # azul medio
    '#70AD47',  # verde
    '#ED7D31',  # naranja
    '#FFC000',  # amarillo
]

# ── Mapeo de nombres cortos ─────────────────────────────────────────────────
NOMBRES_CORTOS = {
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2': 'Mul-384-MiniLM-L12',
    'intfloat/multilingual-e5-small':                              'Mul-384-E5-Small',
    'intfloat/multilingual-e5-base':                               'Mul-768-E5-Base',
    'sentence-transformers/distiluse-base-multilingual-cased-v1':  'Mul-512-Distiluse-v1',
    'sentence-transformers/LaBSE':                                 'Mul-768-LaBSE',
}

# Orden lógico para los gráficos (mejor a peor MRR promedio)
ORDER = ['Mul-768-LaBSE', 'Mul-512-Distiluse-v1', 'Mul-768-E5-Base',
         'Mul-384-E5-Small', 'Mul-384-MiniLM-L12']

print('✅ Entorno configurado. Librerías cargadas correctamente.')

✅ Entorno configurado. Librerías cargadas correctamente.


---
## 1. 📥 Carga y Exploración de Datos

In [2]:
FILE_PATH = 'embedding_comparison_results.xlsx'

df_c = pd.read_excel(FILE_PATH, sheet_name='Resultados_Completos')
df_p = pd.read_excel(FILE_PATH, sheet_name='Resumen_Promedios')

# Aplicar nombres cortos
df_c['Modelo'] = df_c['Modelo'].replace(NOMBRES_CORTOS)
df_p['Modelo'] = df_p['Modelo'].replace(NOMBRES_CORTOS)

# Convertir Overlap a entero para mejor visualización
df_c['Overlap'] = df_c['Overlap'].round().astype(int)

# Calcular métrica compuesta: Eficiencia = MRR / T.Embedding
df_p['Eficiencia'] = df_p['MRR'] / df_p['T. Embedding (s)']

print(f'✅ Datos cargados: {len(df_c)} configuraciones | {len(df_p)} modelos')
print(f'   Chunk sizes evaluados: {sorted(df_c["Chunk Size"].unique())}')
print(f'   Overlaps evaluados:    {sorted(df_c["Overlap"].unique())}')
print()

# Vista del resumen
display(df_p[['Modelo', 'MRR', 'Top_1_Acc', 'Top_3_Acc', 'T. Embedding (s)', 'RAM (MB)']]
        .sort_values('MRR', ascending=False)
        .reset_index(drop=True)
        .style
        .background_gradient(subset=['MRR', 'Top_1_Acc', 'Top_3_Acc'], cmap='YlGn')
        .background_gradient(subset=['T. Embedding (s)'], cmap='YlOrRd')
        .format({'MRR': '{:.4f}', 'Top_1_Acc': '{:.4f}', 'Top_3_Acc': '{:.4f}',
                 'T. Embedding (s)': '{:.3f}', 'RAM (MB)': '{:.0f}'})
)

✅ Datos cargados: 45 configuraciones | 5 modelos
   Chunk sizes evaluados: [np.int64(500), np.int64(1000), np.int64(1500)]
   Overlaps evaluados:    [np.int64(50), np.int64(150), np.int64(300)]



,Modelo,MRR,Top_1_Acc,Top_3_Acc,T. Embedding (s),RAM (MB)
0,Mul-768-LaBSE,0.7797,0.6815,0.8543,1.827,3996
1,Mul-512-Distiluse-v1,0.7088,0.6347,0.7261,0.780,3980
2,Mul-768-E5-Base,0.6864,0.5473,0.7804,2.013,3878
3,Mul-384-E5-Small,0.5980,0.4362,0.6744,0.640,3699
4,Mul-384-MiniLM-L12,0.5579,0.4473,0.5831,0.423,3626


---
## 2. 🏆 Rendimiento de Recuperación Semántica

### Marco teórico: ¿Qué mide el MRR?

El **Mean Reciprocal Rank (MRR)** es la métrica central para evaluar sistemas de recuperación de información:

$$\text{MRR} = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{\text{rank}_i}$$

Donde $\text{rank}_i$ es la posición del fragmento correcto para la consulta $i$. Una puntuación de **1.0** significa que el sistema recupera siempre el fragmento correcto en **primera posición**.

La penalización es exponencial: un resultado en posición 2 vale 0.5, en posición 3 vale 0.33, en posición 10 vale 0.1.

In [3]:
# ── Gráfico 1: MRR promedio con intervalos de confianza ───────────────────
mrr_stats = df_c.groupby('Modelo')['MRR'].agg(['mean', 'std', 'min', 'max']).reset_index()
mrr_stats.columns = ['Modelo', 'MRR_mean', 'MRR_std', 'MRR_min', 'MRR_max']
mrr_stats = mrr_stats.sort_values('MRR_mean', ascending=True)

fig = go.Figure()

for i, row in mrr_stats.iterrows():
    color = PALETTE[ORDER.index(row['Modelo'])]
    fig.add_trace(go.Bar(
        x=[row['MRR_mean']], y=[row['Modelo']],
        orientation='h',
        marker_color=color,
        error_x=dict(type='data', array=[row['MRR_std']], visible=True, color='#888', thickness=2),
        showlegend=False,
        text=f"{row['MRR_mean']:.4f}",
        textposition='outside',
    ))
    # Marcar el máximo con un punto
    fig.add_trace(go.Scatter(
        x=[row['MRR_max']], y=[row['Modelo']],
        mode='markers',
        marker=dict(symbol='diamond', size=10, color=color, line=dict(width=2, color='white')),
        showlegend=False,
        hovertemplate=f"MRR máximo: {row['MRR_max']:.4f}",
        name=''
    ))

fig.add_vline(x=0.7, line_dash='dash', line_color='green', annotation_text='Umbral recomendado (0.70)', annotation_position='top right')

fig.update_layout(
    title='<b>MRR Promedio por Modelo</b> (barras: media ± desv. típica | ◆: máximo alcanzado)',
    xaxis_title='Mean Reciprocal Rank (MRR)',
    yaxis_title='',
    height=420,
    xaxis=dict(range=[0, 1.12]),
    barmode='overlay',
)
fig.show()

In [4]:
# ── Gráfico 2: Comparativa Top-1 vs Top-3 Accuracy ────────────────────────
df_sorted = df_p.sort_values('MRR', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_sorted['Modelo'], y=df_sorted['Top_1_Acc'],
    name='Top-1 Accuracy',
    marker_color='#1F3864',
    text=df_sorted['Top_1_Acc'].map(lambda x: f'{x:.3f}'),
    textposition='inside', textfont=dict(color='white', size=11)
))

fig.add_trace(go.Bar(
    x=df_sorted['Modelo'], y=df_sorted['Top_3_Acc'],
    name='Top-3 Accuracy',
    marker_color='#2E75B6',
    text=df_sorted['Top_3_Acc'].map(lambda x: f'{x:.3f}'),
    textposition='inside', textfont=dict(color='white', size=11)
))

fig.add_trace(go.Scatter(
    x=df_sorted['Modelo'], y=df_sorted['MRR'],
    mode='lines+markers',
    name='MRR (eje secundario)',
    marker=dict(size=10, symbol='circle', color='#FFC000', line=dict(width=2, color='#333')),
    line=dict(color='#FFC000', width=2, dash='dot'),
    yaxis='y2'
))

fig.update_layout(
    title='<b>Precisión Semántica: Top-1 vs Top-3 Accuracy con MRR superpuesto</b>',
    barmode='group',
    height=480,
    yaxis=dict(title='Accuracy', range=[0, 1]),
    yaxis2=dict(title='MRR', overlaying='y', side='right', range=[0, 1], showgrid=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

### 🔍 Interpretación

**Mul-768-LaBSE** domina en todas las métricas de precisión:
- MRR promedio de **0.7797** — el más alto
- Top-3 Accuracy del **85.4%** — en 8 de cada 10 consultas, el fragmento correcto aparece en el top-3
- Consigue **MRR = 1.0** (perfecto) con la configuración óptima chunk_size=500/overlap=50

La brecha entre Top-1 y Top-3 Accuracy revela que los modelos de 384 dimensiones (E5-Small, MiniLM-L12) tienen más dificultad para rankear el fragmento correcto en primera posición, pero mejoran notablemente cuando se amplía el contexto a 3 resultados.

---
## 3. ⏱️ Eficiencia Computacional: El Trade-off Precisión / Velocidad

Un modelo con MRR perfecto puede ser inviable en producción si introduce latencia inaceptable. El análisis coste-beneficio permite identificar el **"sweet spot"**: máxima precisión con mínimo coste computacional.

In [ ]:
# ── Gráfico 3: Bubble chart — MRR vs Tiempo (tamaño = dimensiones del modelo)
dim_map = {
    'Mul-768-LaBSE': 768,
    'Mul-768-E5-Base': 768,
    'Mul-512-Distiluse-v1': 512,
    'Mul-384-E5-Small': 384,
    'Mul-384-MiniLM-L12': 384,
}
df_p_plot = df_p.copy()
df_p_plot['Dimensiones'] = df_p_plot['Modelo'].map(dim_map)

fig = px.scatter(
    df_p_plot,
    x='T. Embedding (s)', y='MRR',
    color='Modelo',
    size='Dimensiones',
    size_max=50,
    text='Modelo',
    color_discrete_sequence=PALETTE,
    title='<b>Trade-off: Calidad vs Velocidad</b><br><sup>Tamaño de burbuja = dimensiones del embedding. Ideal: arriba a la izquierda.</sup>',
    labels={'T. Embedding (s)': 'Tiempo de Embedding (segundos)', 'MRR': 'MRR Promedio'},
    hover_data={'RAM (MB)': ':.0f', 'Eficiencia': ':.3f'}
)

fig.update_traces(textposition='top center', marker=dict(line=dict(width=2, color='white')))
fig.update_traces(textfont=dict(size=10))



# Cuadrantes
fig.add_hline(y=0.70, line_dash='dot', line_color='green', opacity=0.5)
fig.add_vline(x=1.0, line_dash='dot', line_color='orange', opacity=0.5)

# Anotaciones de cuadrantes
fig.add_annotation(x=0.4, y=0.97, text='✅ Zona óptima', showarrow=False,
                   font=dict(color='green', size=11), bgcolor='rgba(200,255,200,0.6)')
fig.add_annotation(x=2.2, y=0.97, text='⚠️ Precisión alta / lento', showarrow=False,
                   font=dict(color='orange', size=11), bgcolor='rgba(255,240,200,0.6)')

fig.update_layout(height=520, showlegend=False)
fig.show()

In [10]:
# ── Gráfico 4: Boxplot de tiempos de embedding por modelo ─────────────────
df_c_ord = df_c.copy()
df_c_ord['Modelo'] = pd.Categorical(df_c_ord['Modelo'], categories=ORDER, ordered=True)
df_c_ord = df_c_ord.sort_values('Modelo')

fig = px.box(
    df_c_ord,
    x='Modelo', y='T. Embedding (s)',
    color='Modelo',
    color_discrete_sequence=PALETTE,
    points='all',
    title='<b>Distribución de Tiempos de Embedding por Modelo</b><br><sup>Estabilidad y varianza a lo largo de las 9 configuraciones de chunking.</sup>'
)
fig.update_layout(height=440, showlegend=False, xaxis_title='')
fig.show()

In [11]:
# ── Gráfico 5: Ranking por eficiencia (MRR / T.Embedding) ─────────────────
df_eff = df_p[['Modelo', 'MRR', 'T. Embedding (s)', 'Eficiencia']].sort_values('Eficiencia', ascending=True)

fig = go.Figure()
for i, row in df_eff.iterrows():
    color = PALETTE[ORDER.index(row['Modelo'])]
    fig.add_trace(go.Bar(
        x=[row['Eficiencia']], y=[row['Modelo']],
        orientation='h',
        marker_color=color,
        text=f"{row['Eficiencia']:.3f}",
        textposition='outside',
        showlegend=False
    ))

fig.update_layout(
    title='<b>Índice de Eficiencia: MRR / Tiempo de Embedding</b><br><sup>Cuántas unidades de precisión se obtienen por segundo de cómputo. Mayor = más eficiente.</sup>',
    xaxis_title='Eficiencia (MRR por segundo de embedding)',
    yaxis_title='',
    height=380
)
fig.show()

### 🔍 Interpretación

El **índice de eficiencia** (MRR / tiempo) revela que **Mul-512-Distiluse-v1** es el modelo más eficiente: obtiene alta precisión en menos tiempo. LaBSE, a pesar de su superior MRR absoluto, tiene la eficiencia más baja debido a su mayor tiempo de cómputo.

**Recomendación por escenario:**
- 🎯 **Máxima precisión**: LaBSE
- ⚡ **Producción / alta carga**: Distiluse-v1  
- 💡 **Edge / recursos limitados**: MiniLM-L12

---
## 4. 🧩 Impacto de la Estrategia de Chunking

La segmentación de documentos es la variable de infraestructura más importante. El mismo modelo puede pasar de un MRR de 0.42 a 1.00 simplemente ajustando el chunk_size.

In [12]:
# ── Gráfico 6: Heatmap global Chunk Size × Overlap ────────────────────────
pivot = df_c.pivot_table(values='MRR', index='Chunk Size', columns='Overlap', aggfunc='mean')

fig = px.imshow(
    pivot,
    text_auto='.3f',
    aspect='auto',
    color_continuous_scale='RdYlGn',
    zmin=0.4, zmax=0.85,
    title='<b>MRR Promedio (todos los modelos) según Chunk Size y Overlap</b>'
)
fig.update_layout(
    xaxis_title='Overlap (caracteres solapados)',
    yaxis_title='Chunk Size (caracteres)',
    coloraxis_colorbar_title='MRR',
    height=400
)
fig.show()

In [13]:
# ── Gráfico 7: Evolución del MRR por chunk size para cada modelo ───────────
mrr_chunk = df_c.groupby(['Modelo', 'Chunk Size'])['MRR'].mean().reset_index()

fig = px.line(
    mrr_chunk,
    x='Chunk Size', y='MRR',
    color='Modelo',
    markers=True,
    color_discrete_sequence=PALETTE,
    title='<b>Evolución del MRR en función del Chunk Size</b><br><sup>Todos los overlaps promediados. Tendencia clara: chunks más pequeños = mejor recuperación.</sup>',
    labels={'Chunk Size': 'Tamaño del Chunk (caracteres)', 'MRR': 'MRR Promedio'},
    category_orders={'Modelo': ORDER}
)
fig.update_traces(line_width=2, marker_size=9)
fig.update_layout(
    height=450,
    xaxis=dict(tickvals=[500, 1000, 1500]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

In [14]:
# ── Gráfico 8: Heatmaps individuales por modelo ────────────────────────────
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[m.replace('Mul-', '') for m in ORDER] + [''],
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for idx, modelo in enumerate(ORDER):
    r = idx // 3 + 1
    c = idx % 3 + 1
    sub = df_c[df_c['Modelo'] == modelo]
    piv = sub.pivot_table(values='MRR', index='Chunk Size', columns='Overlap', aggfunc='mean')
    
    heatmap = go.Heatmap(
        z=piv.values,
        x=[str(v) for v in piv.columns],
        y=[str(v) for v in piv.index],
        colorscale='RdYlGn',
        zmin=0.4, zmax=1.0,
        showscale=(idx == 4),
        text=[[f'{v:.3f}' for v in row] for row in piv.values],
        texttemplate='%{text}',
        textfont=dict(size=11),
    )
    fig.add_trace(heatmap, row=r, col=c)
    fig.update_xaxes(title_text='Overlap' if r == 2 else '', row=r, col=c)
    fig.update_yaxes(title_text='Chunk Size' if c == 1 else '', row=r, col=c)

fig.update_layout(
    title='<b>Mapa de Calor MRR por Modelo</b> (Chunk Size × Overlap)',
    height=640
)
fig.show()

In [15]:
# ── Gráfico 9: Sunburst jerárquico ─────────────────────────────────────────
fig = px.sunburst(
    df_c,
    path=['Modelo', 'Chunk Size', 'Overlap'],
    values='MRR',
    color='MRR',
    color_continuous_scale='RdYlGn',
    title='<b>Desglose Jerárquico del MRR:</b> Modelo → Chunk Size → Overlap'
)
fig.update_layout(height=600, coloraxis_colorbar_title='MRR')
fig.show()

### 🔍 Interpretación del Chunking

El patrón es **universal y consistente** en todos los modelos: el MRR decrece monotónicamente al aumentar el chunk_size.

| Chunk Size | MRR Promedio (todos los modelos) | Explicación |
|:----------:|:--------------------------------:|:------------|
| **500** | ~0.77 | Una unidad conceptual por fragmento. Alta especificidad semántica. |
| 1000 | ~0.65 | Mezcla de 2-3 conceptos. Dilución del vector de embedding. |
| 1500 | ~0.60 | Múltiples conceptos. El embedding pierde foco semántico. |

**Conclusión de chunking**: `chunk_size=500, overlap=50` es la configuración óptima para documentación bancaria estructurada.

---
## 5. 📐 Análisis Multidimensional

In [16]:
# ── Gráfico 10: Radar chart multidimensional ───────────────────────────────
categorias = ['MRR', 'Top-1 Acc', 'Top-3 Acc', 'Velocidad', 'Eficiencia RAM']

# Normalizar las métricas entre 0 y 1
df_norm = df_p.copy()
df_norm['vel_norm']  = 1 - (df_norm['T. Embedding (s)'] - df_norm['T. Embedding (s)'].min()) / (df_norm['T. Embedding (s)'].max() - df_norm['T. Embedding (s)'].min())
df_norm['ram_norm']  = 1 - (df_norm['RAM (MB)'] - df_norm['RAM (MB)'].min()) / (df_norm['RAM (MB)'].max() - df_norm['RAM (MB)'].min())

fig = go.Figure()
for i, row in df_norm.iterrows():
    valores = [
        row['MRR'],
        row['Top_1_Acc'],
        row['Top_3_Acc'],
        row['vel_norm'],
        row['ram_norm'],
    ]
    valores.append(valores[0])  # cerrar el polígono
    cats = categorias + [categorias[0]]
    color = PALETTE[ORDER.index(row['Modelo'])]
    fig.add_trace(go.Scatterpolar(
        r=valores, theta=cats,
        fill='toself', fillcolor=color,
        opacity=0.25,
        line=dict(color=color, width=2),
        name=row['Modelo']
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='<b>Perfil Multidimensional de los Modelos</b><br><sup>Todas las métricas normalizadas entre 0 y 1. Área mayor = mejor rendimiento global.</sup>',
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0.5, xanchor='center'),
    height=520
)
fig.show()

In [17]:
# ── Gráfico 11: Distribución del MRR por modelo (violin plot) ─────────────
fig = go.Figure()
for i, modelo in enumerate(ORDER):
    sub = df_c[df_c['Modelo'] == modelo]['MRR']
    color = PALETTE[i]
    fig.add_trace(go.Violin(
        x=[modelo] * len(sub),
        y=sub,
        name=modelo,
        box_visible=True,
        meanline_visible=True,
        fillcolor=color,
        line_color='white',
        opacity=0.8,
        showlegend=False
    ))

fig.update_layout(
    title='<b>Distribución del MRR: consistencia entre configuraciones</b><br><sup>Un violín estrecho indica robustez: el modelo mantiene la precisión independientemente del chunking.</sup>',
    yaxis_title='MRR',
    xaxis_title='',
    height=450
)
fig.show()

---
## 6. 💡 Conclusiones Estratégicas y Recomendación Final

In [18]:
# ── Tabla resumen de decisión ──────────────────────────────────────────────
print('=' * 75)
print('  RESUMEN DE DECISIÓN: SELECCIÓN DE MODELO DE EMBEDDING')
print('=' * 75)

decisiones = [
    ('🥇 Máxima precisión (prod. baja carga)', 'Mul-768-LaBSE',        'chunk=500, overlap=50', '0.7797', '1.83s'),
    ('🥈 Producción / alta concurrencia',      'Mul-512-Distiluse-v1', 'chunk=500, overlap=50', '0.7088', '0.78s'),
    ('🥉 Recursos limitados / edge',           'Mul-384-MiniLM-L12',   'chunk=500, overlap=50', '0.5579', '0.42s'),
]

print(f"\n{'Escenario':<42} {'Modelo':<24} {'Config.':<22} {'MRR':<8} {'T.Emb'}")
print('-' * 75)
for esc, mod, conf, mrr, t in decisiones:
    print(f"{esc:<42} {mod:<24} {conf:<22} {mrr:<8} {t}")

print()
print('  ✅ RECOMENDACIÓN PRINCIPAL: Mul-768-LaBSE + chunk_size=500 + overlap=50')
print('  MRR máximo alcanzado: 1.0000 (perfecto) — Top-3 Acc media: 85.4%')
print('=' * 75)

  RESUMEN DE DECISIÓN: SELECCIÓN DE MODELO DE EMBEDDING

Escenario                                  Modelo                   Config.                MRR      T.Emb
---------------------------------------------------------------------------
🥇 Máxima precisión (prod. baja carga)      Mul-768-LaBSE            chunk=500, overlap=50  0.7797   1.83s
🥈 Producción / alta concurrencia           Mul-512-Distiluse-v1     chunk=500, overlap=50  0.7088   0.78s
🥉 Recursos limitados / edge                Mul-384-MiniLM-L12       chunk=500, overlap=50  0.5579   0.42s

  ✅ RECOMENDACIÓN PRINCIPAL: Mul-768-LaBSE + chunk_size=500 + overlap=50
  MRR máximo alcanzado: 1.0000 (perfecto) — Top-3 Acc media: 85.4%


In [19]:
# ── Ranking final ponderado ────────────────────────────────────────────────
# Score combinado: 50% MRR + 20% Top1 + 15% Top3 + 15% Velocidad normalizada
df_rank = df_p.copy()
df_rank['vel_score'] = 1 - (df_rank['T. Embedding (s)'] - df_rank['T. Embedding (s)'].min()) / \
                           (df_rank['T. Embedding (s)'].max() - df_rank['T. Embedding (s)'].min())

df_rank['Score_Final'] = (
    0.50 * df_rank['MRR'] +
    0.20 * df_rank['Top_1_Acc'] +
    0.15 * df_rank['Top_3_Acc'] +
    0.15 * df_rank['vel_score']
)
df_rank = df_rank.sort_values('Score_Final', ascending=False).reset_index(drop=True)
df_rank.index += 1

cols_show = ['Modelo', 'MRR', 'Top_1_Acc', 'Top_3_Acc', 'T. Embedding (s)', 'vel_score', 'Score_Final']
df_rank[cols_show].rename(columns={'vel_score': 'Vel. (norm.)', 'Score_Final': '🏆 Score Ponderado'}) \
    .style \
    .background_gradient(subset=['🏆 Score Ponderado'], cmap='YlGn') \
    .format({'MRR': '{:.4f}', 'Top_1_Acc': '{:.4f}', 'Top_3_Acc': '{:.4f}',
             'T. Embedding (s)': '{:.3f}', 'Vel. (norm.)': '{:.3f}', '🏆 Score Ponderado': '{:.4f}'})

,Modelo,MRR,Top_1_Acc,Top_3_Acc,T. Embedding (s),Vel. (norm.),🏆 Score Ponderado
1,Mul-512-Distiluse-v1,0.7088,0.6347,0.7261,0.780,0.776,0.7066
2,Mul-768-LaBSE,0.7797,0.6815,0.8543,1.827,0.117,0.6719
3,Mul-384-E5-Small,0.5980,0.4362,0.6744,0.640,0.864,0.6170
4,Mul-384-MiniLM-L12,0.5579,0.4473,0.5831,0.423,1.000,0.6059
5,Mul-768-E5-Base,0.6864,0.5473,0.7804,2.013,0.000,0.5697


---

## 7. 📋 Resumen Ejecutivo

### ¿Por qué LaBSE?

**LaBSE (Language-Agnostic BERT Sentence Embedding)** fue desarrollado por Google y entrenado con más de **3.000 millones de pares de frases paralelas** en 109 idiomas. Su arquitectura combina:
- *Masked language modeling* bilateral
- *Translation language modeling* para alineación semántica entre idiomas

Esto lo hace especialmente robusto para terminología técnica especializada (como la bancaria) donde el parafraseo y los sinónimos son frecuentes.

### Configuración óptima validada experimentalmente

```
Modelo:     sentence-transformers/LaBSE
Chunk size: 500 caracteres
Overlap:    50 caracteres
Splitter:   RecursiveCharacterTextSplitter
Separadores: ["\n\n", "\n", ".", " ", ""]

MRR:         1.0000 (máximo) | 0.7797 (promedio)
Top-3 Acc:   85.4% (promedio)
T. Embedding: ~1.83s (9 chunks × configuración)
```
